# 00 — Environment Setup (Colab)

Run this notebook **once at the start of every Colab session** to:
1. Verify GPU (A100 preferred)
2. Install PhysicsNeMo + dependencies
3. Mount Google Drive and create checkpoint directories
4. Clone / pull the project repo
5. Smoke-test all critical imports

**CPU sessions are fine** for setup and data pipeline — switch to A100 only for Steps 5–8 (training).

> Runtime → Change runtime type → A100 (only needed for training notebooks)

## 1. GPU check

In [ ]:
import subprocess, sys

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    if 'A100' in result.stdout:
        print('✓  A100 detected — optimal for PhysicsNeMo training.')
    elif 'T4' in result.stdout:
        print('⚠  T4 detected — training will work but slower.')
    elif result.returncode == 0:
        print('GPU detected:')
        print(result.stdout[:300])
    else:
        print('nvidia-smi ran but returned non-zero. Likely no NVIDIA GPU.')
except FileNotFoundError:
    print('No NVIDIA GPU detected (nvidia-smi not found) — CPU session.')
    print('This is fine for 00_env_setup and 01_data_pipeline.')
    print('Switch to A100 GPU before running training notebooks (Steps 5–8).')
except subprocess.TimeoutExpired:
    print('nvidia-smi timed out — assuming CPU session.')

In [ ]:
import torch

HAS_GPU = torch.cuda.is_available()
print(f'PyTorch {torch.__version__}  |  CUDA available: {HAS_GPU}')
if HAS_GPU:
    print(f'CUDA {torch.version.cuda}  |  Device: {torch.cuda.get_device_name(0)}')
else:
    print('CPU session — training notebooks require GPU.')

## 2. Install PhysicsNeMo (GPU sessions only)

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    cuda_ver = torch.version.cuda or ''
    extra = 'cu13' if cuda_ver.startswith('13') else 'cu12'
    print(f'GPU session — installing nvidia-physicsnemo[{extra},nn-extras] ...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', f'nvidia-physicsnemo[{extra},nn-extras]'], check=True)
    print('PhysicsNeMo installed.')
else:
    print('CPU session — skipping PhysicsNeMo install.')
    print('PhysicsNeMo will be installed when you switch to A100 for training.')

In [ ]:
# Install remaining project dependencies (fast, works on CPU and GPU)
import subprocess
subprocess.run(['pip', 'install', '-q',
    'h5py', 'scipy', 'botorch', 'gpytorch',
    'fluidfoam', 'trimesh', 'numpy-stl', 'tqdm', 'pyyaml',
    'airfrans',
], check=True)
subprocess.run(['pip', 'install', '-q',
    'huggingface-hub', 'datasets', 'torch-geometric',
], check=True)
print('Dependencies installed.')

## 3. Mount Google Drive + checkpoint dirs

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
    BASE_DIR = '/content/drive/MyDrive/cfd-ald-app'
    print('Drive mounted.')
except ImportError:
    ON_COLAB = False
    BASE_DIR = os.path.expanduser('~/cfd-ald-app-data')
    print(f'Not on Colab — using local BASE_DIR: {BASE_DIR}')

DIRS = [
    f'{BASE_DIR}/checkpoints/flow_head',
    f'{BASE_DIR}/checkpoints/heat_head',
    f'{BASE_DIR}/checkpoints/species_head',
    f'{BASE_DIR}/checkpoints/multihead',
    f'{BASE_DIR}/data/processed/airfrans',
    f'{BASE_DIR}/data/processed/cfdbench',
    f'{BASE_DIR}/data/processed/showerhead_openfoam',
    f'{BASE_DIR}/logs',
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓ {d}')

print('\nDrive mounted and checkpoint directories ready.')

## 4. Clone / pull project repo

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Repo exists — pulling latest changes...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or 'Already up to date.')
    if result.returncode != 0:
        print('git pull warning:', result.stderr)
else:
    print(f'Cloning {REPO_URL} → {REPO_DIR} ...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode == 0:
        print('Clone successful.')
    else:
        print('Clone failed:', result.stderr)
        raise RuntimeError(f'git clone failed: {result.stderr}')

# Add to Python path so all modules are importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Repo at {REPO_DIR} added to sys.path.')

## 5. Smoke-test imports

PhysicsNeMo is only checked on GPU sessions — it is not installed on CPU.

In [ ]:
import importlib, traceback, torch

HAS_GPU = torch.cuda.is_available()

# Core checks always run
checks_always = [
    ('torch',              lambda m: f'PyTorch {m.__version__}'),
    ('h5py',               lambda m: f'h5py {m.__version__}'),
    ('scipy',              lambda m: f'SciPy {m.__version__}'),
    ('botorch',            lambda m: f'BoTorch {m.__version__}'),
    ('airfrans',           lambda m: f'airfrans {getattr(m, "__version__", "ok")}'),
    ('physics.calculator', lambda m: 'calculator OK'),
    ('physics.guardrails', lambda m: 'guardrails OK'),
]

# GPU-only checks
checks_gpu = [
    ('physicsnemo',        lambda m: f'PhysicsNeMo {m.__version__}'),
    ('physicsnemo.models', lambda m: 'models OK'),
    ('torch_geometric',    lambda m: f'PyG {m.__version__}'),
]

all_ok = True

print('── Always-required imports ──')
for mod_name, desc_fn in checks_always:
    try:
        m = importlib.import_module(mod_name)
        print(f'  ✓  {desc_fn(m)}')
    except Exception:
        print(f'  ✗  {mod_name}')
        traceback.print_exc()
        all_ok = False

if HAS_GPU:
    print('\n── GPU-only imports ──')
    for mod_name, desc_fn in checks_gpu:
        try:
            m = importlib.import_module(mod_name)
            print(f'  ✓  {desc_fn(m)}')
        except Exception:
            print(f'  ✗  {mod_name}')
            traceback.print_exc()
            all_ok = False
else:
    print('\n── GPU-only imports skipped (CPU session) ──')
    print('  (PhysicsNeMo, torch-geometric — install when switching to A100)')

print()
print('Environment ready ✓' if all_ok else 'Some imports failed — check above.')

## 6. Quick physics calculator sanity check

In [ ]:
from physics.calculator import compute_all, k_rxn_from_sticking
from physics.guardrails import GuardrailEngine

# Example: N2 carrier gas, 2mm nozzle, 5 m/s exit velocity, TMA precursor
params = {
    'rho':     1.12,
    'V':       5.0,
    'L':       0.002,
    'mu':      2.0e-5,
    'cp':      1040.0,
    'k_fluid': 0.031,
    'D_m':     2.5e-5,
    'a':       380.0,
    'k_rxn':   k_rxn_from_sticking(beta=0.05, v_th=144.0),
}

numbers = compute_all(params)
print('Dimensionless numbers for example ALD showerhead case:')
print('-' * 45)
for k, v in sorted(numbers.items()):
    print(f'  {k:6s} = {v:.4g}')

engine = GuardrailEngine()
result = engine.check(numbers)
print()
print(result.summary())

## 7. Save session config

Saves paths to Drive so other notebooks can find them without re-entering.

In [ ]:
import json, torch
from pathlib import Path

session_cfg = {
    'base_dir':     BASE_DIR,
    'repo_dir':     REPO_DIR,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
    'gpu_name':     torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'cuda_version': torch.version.cuda,
}

cfg_path = Path(BASE_DIR) / 'session_config.json'
cfg_path.write_text(json.dumps(session_cfg, indent=2))

print(f'Session config saved to {cfg_path}')
print(json.dumps(session_cfg, indent=2))
print('\n✓  Run 01_data_pipeline.ipynb next.')